In [2]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np

In [4]:
def cosine_similarity(a,b):
    a = np.array(a)
    b= np.array(b)
    ab= a@b
    na= np.linalg.norm(a)
    nb= np.linalg.norm(b)
    return float(ab/(na*nb))

In [6]:
cat= [0.8,0.6,0.3]
dog= [0.75,0.65,0.35]
car= [-0.5,0.2,0.9]

In [7]:
print("Cosine similarity between cat and dog:", cosine_similarity(cat,dog))
print("Cosine similarity between cat and car:", cosine_similarity(cat,car))

Cosine similarity between cat and dog: 0.9966186334192181
Cosine similarity between cat and car: -0.00913251529954447


In [10]:
pip install -U sentence-transformers

  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.12.1-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.26.8-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached shellingha

In [11]:
pip install -U langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\hp\Desktop\Vector_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3483.42it/s]


In [14]:
query= "I love Programming"
queryEmbedding= embeddings.embed_query(query)

In [15]:
len(queryEmbedding)

384

In [17]:
documents=[
    "The stock market experienced a significant downturn today, with major indices falling by over 2%.",
    "Scientists have discovered a new species of frog in the Amazon rainforest, which has unique color patterns and behaviors.",
    "The chef carefully folded the egg whites into the batter, ensuring a light and fluffy texture for the soufflé.",
    "After three hours of delibiration, the jury reached a unanimous verdict, finding the defendant guilty on all counts.",
    "Migrating birds often travel thousands of miles each year, navigating using the Earth's magnetic field and the position of the sun.",
    "I am learning Langchain"
]

In [18]:
docs_vectors= embeddings.embed_documents(documents)

In [19]:
query= "what was the verdict of the jury?"
query_vector= embeddings.embed_query(query)

In [20]:
scores=[]
for index, doc_vector in enumerate(docs_vectors):
    score= cosine_similarity(query_vector, doc_vector)
    scores.append({"document": documents[index], "score": score})
    

In [21]:
scores.sort(key=lambda x: x["score"], reverse=True)
scores[:2]

[{'document': 'After three hours of delibiration, the jury reached a unanimous verdict, finding the defendant guilty on all counts.',
  'score': 0.6552964808209062},
 {'document': 'The chef carefully folded the egg whites into the batter, ensuring a light and fluffy texture for the soufflé.',
  'score': 0.06467770248082214}]

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore

In [23]:
documents=[
    "The stock market experienced a significant downturn today, with major indices falling by over 2%.",
    "Scientists have discovered a new species of frog in the Amazon rainforest, which has unique color patterns and behaviors.",
    "The chef carefully folded the egg whites into the batter, ensuring a light and fluffy texture for the soufflé.",
    "After three hours of delibiration, the jury reached a unanimous verdict, finding the defendant guilty on all counts.",
    "Migrating birds often travel thousands of miles each year, navigating using the Earth's magnetic field and the position of the sun.",
    "I am learning Langchain"
]

In [24]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6503.78it/s]


In [26]:
vector_store = InMemoryVectorStore.from_texts(texts=documents, embedding=embeddings)

In [27]:
records= vector_store.similarity_search(query="what was the verdict of the jury?", k=2)

In [28]:
records

[Document(id='637b5118-e17c-449d-84d1-957acc0eee6c', metadata={}, page_content='After three hours of delibiration, the jury reached a unanimous verdict, finding the defendant guilty on all counts.'),
 Document(id='ce6bfe59-fbee-45d6-ad94-adf5c9fd2efd', metadata={}, page_content='The chef carefully folded the egg whites into the batter, ensuring a light and fluffy texture for the soufflé.')]

In [29]:
vector_store.dump("./data/vector_store.json")

In [52]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda

load_dotenv()
print("Packages imported...")

Packages imported...


In [53]:
pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [54]:
loader= PyMuPDFLoader(file_path="../data/pdf/medical_report.pdf")
docs= loader.load()

In [55]:
splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data= splitter.split_documents(docs)

In [56]:
embeddings= HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(documents=splitter_data,embedding=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6311.28it/s]


In [57]:

prompt = [
    {"role":"system", "content":"You are a helpful assistant, you always answer based on the provided context, if you not find the relevent answer in the context, then just say, 'I don't have enough information'"},
    {"role":"user", "content":"Here is the context {context}, and my question is: {query}"}
]

prompt = ChatPromptTemplate.from_messages(prompt)

In [58]:
llm= ChatOpenAI(model="gpt-4.1")

In [59]:
def getContext(query:str):
    similar_records = vector_store.similarity_search(query=query, k = 2)
    # llm - [:4]
    context = ""
    for chunk in similar_records:
        context += chunk.page_content + "\n"
        
    ## get exect match data. 
    ## Re-Rankar. 
    return {"context": context, "query": query}

getContext = RunnableLambda(getContext)

In [60]:
rag_chain = getContext | prompt | llm

In [61]:
query = "What is the value of Hemoglobin, and value of 'ANTI CCP' test"
response = rag_chain.invoke(query)

In [62]:

print(response.content)

Based on the provided context:

- The value of Hemoglobin is **13.10 g/dL**.
- The value of the **ANTI CCP (Cyclic Citrullinated Peptide)** test is **<8.0 U/mL**.
